# 02 — Canonical Schema and Neo4j Graph Construction

This notebook converts the prepared PaySim modeling dataset into a canonical
transaction schema and builds a Neo4j graph representation.

Graph structure:

`Account → SENT → Transaction → TO → Account`

## 1. Load the Prepared Modeling Dataset

In [1]:
import pandas as pd

df_model = pd.read_csv("../data/paysim_model_200k.csv")

print("Dataset shape:", df_model.shape)
df_model.head()

Dataset shape: (200000, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,207,CASH_OUT,201544.43,C1446786192,0.0,0.0,C629162506,4427398.04,4628942.46,0,0
1,301,CASH_OUT,99796.32,C1507137317,0.0,0.0,C95898629,2244500.63,2344296.95,0,0
2,429,CASH_OUT,31901.99,C689120377,0.0,0.0,C1777573992,9080459.95,9112361.94,0,0
3,187,CASH_OUT,286477.31,C1224308913,0.0,0.0,C766410750,949477.95,1235955.26,0,0
4,159,CASH_OUT,119075.98,C2098350559,0.0,0.0,C85657936,187137.64,306213.62,0,0


## 2. Convert PaySim Columns to a Canonical Schema

PaySim-specific column names are mapped to generic transaction fields.
This provides a common structure for graph construction and later modeling stages.

In [2]:
df_canonical = df_model.rename(columns={
    "nameOrig": "src_id",
    "nameDest": "dst_id",
    "step": "time_step",
    "type": "tx_type",
    "isFraud": "label_fraud",
    "isFlaggedFraud": "label_flagged",
    "oldbalanceOrg": "src_balance_before",
    "newbalanceOrig": "src_balance_after",
    "oldbalanceDest": "dst_balance_before",
    "newbalanceDest": "dst_balance_after"
}).copy()

df_canonical.insert(
    0,
    "tx_id",
    [f"tx_{i}" for i in range(1, len(df_canonical) + 1)]
)

print("Canonical shape:", df_canonical.shape)

print("\nColumns:")
print(df_canonical.columns.tolist())

df_canonical.head()

Canonical shape: (200000, 12)

Columns:
['tx_id', 'time_step', 'tx_type', 'amount', 'src_id', 'src_balance_before', 'src_balance_after', 'dst_id', 'dst_balance_before', 'dst_balance_after', 'label_fraud', 'label_flagged']


,tx_id,time_step,tx_type,amount,src_id,src_balance_before,src_balance_after,dst_id,dst_balance_before,dst_balance_after,label_fraud,label_flagged
0,tx_1,207,CASH_OUT,201544.43,C1446786192,0.0,0.0,C629162506,4427398.04,4628942.46,0,0
1,tx_2,301,CASH_OUT,99796.32,C1507137317,0.0,0.0,C95898629,2244500.63,2344296.95,0,0
2,tx_3,429,CASH_OUT,31901.99,C689120377,0.0,0.0,C1777573992,9080459.95,9112361.94,0,0
3,tx_4,187,CASH_OUT,286477.31,C1224308913,0.0,0.0,C766410750,949477.95,1235955.26,0,0
4,tx_5,159,CASH_OUT,119075.98,C2098350559,0.0,0.0,C85657936,187137.64,306213.62,0,0


> `tx_id` is a generated unique identifier and does not represent chronological order.
> Transaction chronology is determined using `time_step`.

In [3]:
df_canonical.to_csv(
    "../data/paysim_canonical_200k.csv",
    index=False
)

print("Saved: ../data/paysim_canonical_200k.csv")

Saved: ../data/paysim_canonical_200k.csv


## 3. Build Graph Tables

The graph contains two node types:

- `Account`
- `Transaction`

and two directed relationships:

- `SENT`
- `TO`

Each transaction is represented as:

`(:Account)-[:SENT]->(:Transaction)-[:TO]->(:Account)`

In [4]:
accounts = pd.DataFrame({
    "account_id": pd.concat(
        [
            df_canonical["src_id"],
            df_canonical["dst_id"]
        ],
        ignore_index=True
    ).unique()
})

print("Account table shape:", accounts.shape)
print(
    "Duplicate account IDs:",
    accounts["account_id"].duplicated().sum()
)

accounts.head()

Account table shape: (347598, 1)
Duplicate account IDs: 0


,account_id
0,C1446786192
1,C1507137317
2,C689120377
3,C1224308913
4,C2098350559


In [5]:
transactions = df_canonical.copy()

print("Transaction table shape:", transactions.shape)

print(
    "Unique transaction IDs:",
    transactions["tx_id"].nunique()
)

print(
    "Duplicate transaction IDs:",
    transactions["tx_id"].duplicated().sum()
)

transactions.head()

Transaction table shape: (200000, 12)
Unique transaction IDs: 200000
Duplicate transaction IDs: 0


,tx_id,time_step,tx_type,amount,src_id,src_balance_before,src_balance_after,dst_id,dst_balance_before,dst_balance_after,label_fraud,label_flagged
0,tx_1,207,CASH_OUT,201544.43,C1446786192,0.0,0.0,C629162506,4427398.04,4628942.46,0,0
1,tx_2,301,CASH_OUT,99796.32,C1507137317,0.0,0.0,C95898629,2244500.63,2344296.95,0,0
2,tx_3,429,CASH_OUT,31901.99,C689120377,0.0,0.0,C1777573992,9080459.95,9112361.94,0,0
3,tx_4,187,CASH_OUT,286477.31,C1224308913,0.0,0.0,C766410750,949477.95,1235955.26,0,0
4,tx_5,159,CASH_OUT,119075.98,C2098350559,0.0,0.0,C85657936,187137.64,306213.62,0,0


In [6]:
sent_relationships = (
    df_canonical[["src_id", "tx_id"]]
    .rename(columns={
        "src_id": "from_account_id",
        "tx_id": "to_tx_id"
    })
    .copy()
)

to_relationships = (
    df_canonical[["tx_id", "dst_id"]]
    .rename(columns={
        "tx_id": "from_tx_id",
        "dst_id": "to_account_id"
    })
    .copy()
)

print("SENT relationships:", sent_relationships.shape)
print("TO relationships:", to_relationships.shape)

display(sent_relationships.head())
display(to_relationships.head())

SENT relationships: (200000, 2)
TO relationships: (200000, 2)


,from_account_id,to_tx_id
0,C1446786192,tx_1
1,C1507137317,tx_2
2,C689120377,tx_3
3,C1224308913,tx_4
4,C2098350559,tx_5


,from_tx_id,to_account_id
0,tx_1,C629162506
1,tx_2,C95898629
2,tx_3,C1777573992
3,tx_4,C766410750
4,tx_5,C85657936


In [7]:
print(
    "Duplicate SENT transaction IDs:",
    sent_relationships["to_tx_id"].duplicated().sum()
)

print(
    "Duplicate TO transaction IDs:",
    to_relationships["from_tx_id"].duplicated().sum()
)

Duplicate SENT transaction IDs: 0
Duplicate TO transaction IDs: 0


## 4. Save Graph Tables

In [8]:
accounts.to_csv(
    "../data/accounts.csv",
    index=False
)

transactions.to_csv(
    "../data/transactions.csv",
    index=False
)

sent_relationships.to_csv(
    "../data/sent_relationships.csv",
    index=False
)

to_relationships.to_csv(
    "../data/to_relationships.csv",
    index=False
)

print("Saved graph tables:")
print("accounts:", accounts.shape)
print("transactions:", transactions.shape)
print("sent_relationships:", sent_relationships.shape)
print("to_relationships:", to_relationships.shape)

Saved graph tables:
accounts: (347598, 1)
transactions: (200000, 12)
sent_relationships: (200000, 2)
to_relationships: (200000, 2)


## 5. Connect to Neo4j

The Neo4j Python driver is used to load the graph tables into a local database.
Credentials are requested at runtime instead of being stored in the notebook.

In [9]:
from neo4j import GraphDatabase
from getpass import getpass

URI = "neo4j://127.0.0.1:7687"
USER = "neo4j"
DATABASE = "neo4j"

PASSWORD = getpass("Neo4j password: ")

driver = GraphDatabase.driver(
    URI,
    auth=(USER, PASSWORD)
)

driver.verify_connectivity()

print("Neo4j connection successful.")

Neo4j password:  ········


Neo4j connection successful.


## 6. Create Node Constraints

Unique constraints ensure that each account and transaction is represented
by a single node in the graph.

In [10]:
with driver.session(database=DATABASE) as session:

    session.run("""
    CREATE CONSTRAINT account_id_unique IF NOT EXISTS
    FOR (a:Account)
    REQUIRE a.account_id IS UNIQUE
    """)

    session.run("""
    CREATE CONSTRAINT transaction_id_unique IF NOT EXISTS
    FOR (t:Transaction)
    REQUIRE t.tx_id IS UNIQUE
    """)

print("Neo4j constraints ready.")

Neo4j constraints ready.


In [11]:
def chunk_df(df, size=5000):
    for start in range(0, len(df), size):
        yield df.iloc[start:start + size]

In [12]:
def import_accounts(tx, rows):
    query = """
    UNWIND $rows AS row

    MERGE (a:Account {
        account_id: row.account_id
    })
    """

    tx.run(query, rows=rows)

In [13]:
with driver.session(database=DATABASE) as session:

    for i, chunk in enumerate(
        chunk_df(accounts, size=5000),
        start=1
    ):
        rows = chunk.to_dict("records")

        session.execute_write(
            import_accounts,
            rows
        )

        if i % 10 == 0:
            print(f"{i}. account batch completed")

print("Account import completed.")

10. account batch completed
20. account batch completed
30. account batch completed
40. account batch completed
50. account batch completed
60. account batch completed
70. account batch completed
Account import completed.


In [14]:
with driver.session(database=DATABASE) as session:

    result = session.run("""
    MATCH (a:Account)
    RETURN count(a) AS account_count
    """)

    account_count = result.single()["account_count"]

print("Account nodes:", account_count)

Account nodes: 347598


## 7. Import Transaction Nodes

Each row in the canonical transaction table becomes a `Transaction` node
with transaction-level properties.

In [15]:
def import_transactions(tx, rows):

    query = """
    UNWIND $rows AS row

    MERGE (t:Transaction {
        tx_id: row.tx_id
    })

    SET
        t.time_step = row.time_step,
        t.tx_type = row.tx_type,
        t.amount = row.amount,
        t.src_balance_before = row.src_balance_before,
        t.src_balance_after = row.src_balance_after,
        t.dst_balance_before = row.dst_balance_before,
        t.dst_balance_after = row.dst_balance_after,
        t.label_fraud = row.label_fraud,
        t.label_flagged = row.label_flagged
    """

    tx.run(query, rows=rows)

In [16]:
with driver.session(database=DATABASE) as session:

    for i, chunk in enumerate(
        chunk_df(transactions, size=5000),
        start=1
    ):
        rows = chunk.to_dict("records")

        session.execute_write(
            import_transactions,
            rows
        )

        if i % 10 == 0:
            print(f"{i}. transaction batch completed")

print("Transaction import completed.")

10. transaction batch completed
20. transaction batch completed
30. transaction batch completed
40. transaction batch completed
Transaction import completed.


In [ ]:
with driver.session(database=DATABASE) as session:

    result = session.run("""
    MATCH (t:Transaction)
    RETURN count(t) AS tx_count
    """)

    tx_count = result.single()["tx_count"]

print("Transaction nodes:", tx_count)

## 8. Import Graph Relationships

`SENT` connects the source account to the transaction.

`TO` connects the transaction to the destination account.

In [17]:
def import_sent_relationships(tx, rows):

    query = """
    UNWIND $rows AS row

    MATCH (a:Account {
        account_id: row.from_account_id
    })

    MATCH (t:Transaction {
        tx_id: row.to_tx_id
    })

    MERGE (a)-[:SENT]->(t)
    """

    tx.run(query, rows=rows)

In [18]:
with driver.session(database=DATABASE) as session:

    for i, chunk in enumerate(
        chunk_df(sent_relationships, size=5000),
        start=1
    ):
        rows = chunk.to_dict("records")

        session.execute_write(
            import_sent_relationships,
            rows
        )

        if i % 10 == 0:
            print(f"{i}. SENT batch completed")

print("SENT relationship import completed.")

10. SENT batch completed
20. SENT batch completed
30. SENT batch completed
40. SENT batch completed
SENT relationship import completed.


In [19]:
def import_to_relationships(tx, rows):

    query = """
    UNWIND $rows AS row

    MATCH (t:Transaction {
        tx_id: row.from_tx_id
    })

    MATCH (a:Account {
        account_id: row.to_account_id
    })

    MERGE (t)-[:TO]->(a)
    """

    tx.run(query, rows=rows)

In [20]:
with driver.session(database=DATABASE) as session:

    for i, chunk in enumerate(
        chunk_df(to_relationships, size=5000),
        start=1
    ):
        rows = chunk.to_dict("records")

        session.execute_write(
            import_to_relationships,
            rows
        )

        if i % 10 == 0:
            print(f"{i}. TO batch completed")

print("TO relationship import completed.")

10. TO batch completed
20. TO batch completed
30. TO batch completed
40. TO batch completed
TO relationship import completed.


## 9. Validate the Graph

The final graph should contain one `SENT` and one `TO` relationship
for every transaction.

In [21]:
with driver.session(database=DATABASE) as session:

    result = session.run("""
    MATCH (a:Account)
    WITH count(a) AS account_count

    MATCH (t:Transaction)
    WITH account_count, count(t) AS transaction_count

    MATCH ()-[s:SENT]->()
    WITH account_count,
         transaction_count,
         count(s) AS sent_count

    MATCH ()-[r:TO]->()

    RETURN
        account_count,
        transaction_count,
        sent_count,
        count(r) AS to_count
    """)

    graph_summary = dict(result.single())

graph_summary

{'account_count': 347598,
 'transaction_count': 200000,
 'sent_count': 200000,
 'to_count': 200000}

In [22]:
with driver.session(database=DATABASE) as session:

    result = session.run("""
    MATCH (src:Account)-[:SENT]->
          (t:Transaction)-[:TO]->
          (dst:Account)

    WHERE t.label_fraud = 1

    RETURN
        src.account_id AS sender,
        t.tx_id AS transaction_id,
        t.tx_type AS transaction_type,
        t.amount AS amount,
        dst.account_id AS receiver

    LIMIT 5
    """)

    fraud_examples = pd.DataFrame(
        [dict(record) for record in result]
    )

fraud_examples

,sender,transaction_id,transaction_type,amount,receiver
0,C1149220913,tx_10,CASH_OUT,3244067.13,C771492028
1,C2058293916,tx_25,CASH_OUT,23134.17,C584398732
2,C790582807,tx_26,CASH_OUT,4417191.86,C690378183
3,C278936084,tx_32,TRANSFER,1158070.88,C1115591594
4,C263309564,tx_34,CASH_OUT,42513.08,C1511183581


In [23]:
driver.close()